In [42]:
import pandas as pd
import numpy as np

df = pd.read_csv("Online Retail.csv")

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01/12/2010 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,01/12/2010 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01/12/2010 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01/12/2010 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01/12/2010 08:26,3.39,17850.0,United Kingdom


In [43]:
#Revisamos la información del dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [44]:
#Revisamos la cantidad de filas y columnas del dataset.
df.shape

(541909, 8)

In [45]:
df['CustomerID'].nunique()

4372

In [46]:
df['InvoiceNo'].nunique()

25900

In [47]:
#Eliminamos los nulos de clientes, ya que, como analizaremos la retención de clientes, si el ID es nulo, estas columnas no nos servirán para el análisis.
df = df.dropna(subset=['CustomerID'])

In [48]:
df['InvoiceNo'].str.startswith('C').sum()

np.int64(8905)

In [49]:
#Ya que analizaremos la retención de clientes, eliminamos las cancelaciones de pedido del datset.
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

In [50]:
#Eliminamos los valores inválidos del dataset.
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

In [51]:
#Transformamos el InvoiceDate desde object a datetime.
#Transformamos el CustomerID desde float a int.
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], dayfirst=True)
df['CustomerID'] = df['CustomerID'].astype(int)

In [52]:
#Creamos nueva métrica.
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

In [53]:
#Revisamos duplicados
df.duplicated().sum()

np.int64(5192)

In [54]:
df = df.drop_duplicates()

In [55]:
#Buscamos Outliers
df['Quantity'].describe()

count    392692.000000
mean         13.119702
std         180.492832
min           1.000000
25%           2.000000
50%           6.000000
75%          12.000000
max       80995.000000
Name: Quantity, dtype: float64

In [56]:
#Buscamos Outliers
df['UnitPrice'].describe()

count    392692.000000
mean          3.125914
std          22.241836
min           0.001000
25%           1.250000
50%           1.950000
75%           3.750000
max        8142.750000
Name: UnitPrice, dtype: float64

In [57]:
#Eliminamos outliers, aplicando percentiles (P99), eliminando el 1% de los datos más altos, y así eliminando el sesgo de los datos en cantidad ordenada y precio unitario.
q_high = df['Quantity'].quantile(0.99)
p_high = df['UnitPrice'].quantile(0.99)

df = df[(df['Quantity'] <= q_high) & (df['UnitPrice'] <= p_high)]

In [58]:
#Chequeamos resultado de nuestra limpieza de outliers
df['Quantity'].describe()

count    385081.00000
mean         10.00891
std          14.64235
min           1.00000
25%           2.00000
50%           6.00000
75%          12.00000
max         120.00000
Name: Quantity, dtype: float64

In [59]:
#Chequeamos resultado de nuestra limpieza de outliers
df['UnitPrice'].describe()

count    385081.000000
mean          2.731160
std           2.545555
min           0.001000
25%           1.250000
50%           1.950000
75%           3.750000
max          14.950000
Name: UnitPrice, dtype: float64

In [60]:
df.shape

(385081, 9)

In [61]:
#Revisión de valores nulos post limpieza
df.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
TotalPrice     0
dtype: int64

In [62]:
df['CustomerID'].nunique()

4290

In [63]:
df['InvoiceNo'].nunique()

17980

In [64]:
df['TotalPrice'].sum()

np.float64(6896447.284)

In [65]:
#Se eliminaron aprox 100 registros en CustomerID, lo cual indica una limpieza correcta.
#Se eliminaron aproximadamente 9000 registros en InvoiceNo, lo cual inicialmente puede parecer alarmante, sin embargo, la mayoría de estos registros eliminados correspondían a cancelaciones (8900 aproximadamente), por lo que se justifica la limpieza agresiva del dataset, puesto que estamos evaluando retención de clientes.

In [66]:
df.shape

(385081, 9)

In [67]:
df.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
TotalPrice     0
dtype: int64

In [68]:
#Determinamos que el dataset se encuentra saludable y limpio, listo para realizar el análisis de retención de clientes.

In [69]:
#Creamos columna mes.
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

In [70]:
#Creamos cohortes para cada cliente (en nuestro caso, el mes de la primera compra de cada cliente)
df['CohortMonth'] = df.groupby('CustomerID')['InvoiceDate'] \
                      .transform('min') \
                      .dt.to_period('M')

In [71]:
#Creamos índice de cohorte (tiempo pasado en meses desde x compra hasta la primera compra de dicho cliente, x = compra de la fila actual)
df['CohortIndex'] = (df['InvoiceMonth'] - df['CohortMonth']).apply(lambda x: x.n)

In [72]:
df[['CustomerID', 'InvoiceMonth', 'CohortMonth', 'CohortIndex']].sample(5)

,CustomerID,InvoiceMonth,CohortMonth,CohortIndex
230372,13090,2011-06,2010-12,6
119107,15311,2011-03,2010-12,3
323588,12437,2011-09,2011-01,8
130087,17705,2011-03,2011-02,1
130440,17507,2011-03,2011-02,1


In [73]:
#Un cliente que compra varias veces en el mismo mes, generaría un registro de cohorte por cada transacción, lo que generaría inflación en la retención.
#Corregimos el dataset a que cada cliente sea contado 1 sola vez por período.
df_cohort = df.groupby(['CohortMonth', 'CohortIndex', 'CustomerID']) \
              .agg({'InvoiceNo': 'nunique'}) \
              .reset_index()

In [74]:
df_cohort.head()

,CohortMonth,CohortIndex,CustomerID,InvoiceNo
0,2010-12,0,12347,1
1,2010-12,0,12348,1
2,2010-12,0,12370,2
3,2010-12,0,12377,1
4,2010-12,0,12383,1


In [75]:
#Contamos cuantos clientes únicos hay por cohorte y por mes.
cohort_data = df_cohort.groupby(['CohortMonth', 'CohortIndex']) \
                       .agg(Customers=('CustomerID', 'nunique')) \
                       .reset_index()

In [76]:
cohort_data.head()

,CohortMonth,CohortIndex,Customers
0,2010-12,0,868
1,2010-12,1,317
2,2010-12,2,277
3,2010-12,3,327
4,2010-12,4,315


In [77]:
#Obtenemos tamaño de cohortes
cohort_sizes = cohort_data[cohort_data['CohortIndex'] == 0] \
                .set_index('CohortMonth')['Customers']

In [78]:
#Calculamos la retención
cohort_data['Retention'] = cohort_data.apply(
    lambda x: x['Customers'] / cohort_sizes[x['CohortMonth']],
    axis=1
)

In [79]:
cohort_data.head()

,CohortMonth,CohortIndex,Customers,Retention
0,2010-12,0,868,1.000000
1,2010-12,1,317,0.365207
2,2010-12,2,277,0.319124
3,2010-12,3,327,0.376728
4,2010-12,4,315,0.362903


In [80]:
#Creamos la matriz de cohortes
cohort_pivot = cohort_data.pivot(
    index='CohortMonth',
    columns='CohortIndex',
    values='Retention'
)

In [81]:
#La tabla muestra qué porcentaje de clientes vuelve a comprar con el paso del tiempo desde su primera compra.
#Cada fila representa un cohorte diferente, es decir, meses donde clientes compraron por primera vez (ej: clientes que compraron por primera vez en diciembre de 2010, clientes que compraron por primera vez en enero de 2011).
#Cada columna representa el índice del cohorte, es decir, meses que pasaron después de dicho cohorte, o más bien, meses que pasaron después de la primera compra de los clientes de dicho cohorte.
cohort_pivot.head()

CohortIndex,0,1,2,3,4,5,6,7,8,9,10,11,12
CohortMonth,,,,,,,,,,,,,
2010-12,1.0,0.365207,0.319124,0.376728,0.362903,0.400922,0.361751,0.344470,0.346774,0.391705,0.372120,0.504608,0.262673
2011-01,1.0,0.221411,0.274939,0.226277,0.318735,0.289538,0.253041,0.245742,0.304136,0.330900,0.362530,0.119221,NaN
2011-02,1.0,0.184492,0.187166,0.278075,0.270053,0.243316,0.248663,0.270053,0.248663,0.310160,0.066845,NaN,NaN
2011-03,1.0,0.146341,0.246120,0.201774,0.223947,0.166297,0.263858,0.230599,0.274945,0.084257,NaN,NaN,NaN
2011-04,1.0,0.213559,0.206780,0.206780,0.189831,0.230508,0.220339,0.261017,0.074576,NaN,NaN,NaN,NaN


In [82]:
cohort_data.to_csv('cohort_retention_long.csv', index=False)